# Dados aeroportuários auditáveis — seis experimentos com a pista do SBSJ

**Pergunta.** Uma arquitetura baseada em *event sourcing* garante que dados aeroportuários geolocalizados sejam **rastreáveis até a causa**, **recalculáveis a partir das premissas**, **reconstruíveis no tempo**, com **adulteração evidente** e **consistentes entre produtos**, também para **quem lê sem acesso ao banco**? Onde estão os limites?

Cada experimento testa uma dessas propriedades. A resposta é dada pelo código: os `assert` no fim do notebook dizem sim ou não.

Os experimentos usam a biblioteca [`aeroledger`](https://github.com/heyfalheiro1537/aeroledger) como qualquer usuário usaria: cada passo é uma chamada a ela.

| | Propriedade | Pergunta |
|---|---|---|
| **E1** | causa (caracterização) | O registro das edições publicadas, que guarda só o valor final de cada uma, registra por que um valor mudou? |
| **E2** | derivação | Guardando só as premissas, os valores publicados se recalculam dentro da precisão do AIP? E trocar uma premissa obriga a reescrever algum outro dado? |
| **E3** | reconstrução no tempo | O histórico guardado basta para reconstruir o estado de qualquer instante, com autor, motivo e causa de cada mudança? |
| **E4** | evidência de adulteração | Uma alteração feita direto no banco, sem passar pela biblioteca, é detectada? Qual escapa, e o que a detecta? |
| **E5** | consistência e verificação | Dois produtos gerados do mesmo banco podem divergir? Um terceiro consegue verificar a integridade sem usar a biblioteca? |
| **E6** | leitura separada | Quem lê por um tópico, sem acesso ao banco, chega à mesma resposta, nunca vê uma mudança pela metade e percebe um canal adulterado? Qual adulteração escapa? |

**Como ler.** Cada experimento mostra os resultados. As verificações — o que se esperava, com que margem e por quê — ficam juntas no fim, em células separadas, para que os resultados apareçam mesmo que alguma verificação falhe.

Fontes: AIP Brasil, AD 2.12 e AD 2.13 (eAIP 20-2026); Carta de Aeródromo AIRAC 2604A1; Carta de Obstáculos AIRAC 2213A1. Uso em pesquisa: nada aqui substitui a publicação oficial.

In [ ]:
import json
import tempfile
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import aeroledger as al
from aeroledger import adc, derivacoes
from aeroledger.referencial import Resolvedor

import apoio
import ataques
import fontes as F
import geodesia as gabarito                  # outro código, sem a biblioteca
import verificador_independente as vi        # idem
from apoio import seed as s

def tabela(cabecalho, linhas):
    display(Markdown("| " + " | ".join(cabecalho) + " |\n|" + "---|" * len(cabecalho) + "\n" +
                     "\n".join("| " + " | ".join(str(c) for c in l) + " |" for l in linhas)))

conn = apoio.banco_novo()
print("aeroledger", al.__version__, "· banco", apoio.BANCO, "recriado vazio")

## E1 — Causa (caracterização): o registro das edições publicadas diz por que um valor mudou?

A caracterização vem primeiro. A pergunta é sobre o que chega hoje ao usuário, não sobre o modelo proposto: se a resposta for não, há um problema a resolver. Não é um teste do AIXM completo, que tem *TimeSlices*, números de correção e anotações; é o registro das duas edições tal como o usuário as recebe.

A pista é registrada **como as edições chegam ao usuário**: cada número publicado vira um campo guardado. Um registro com a Carta de Obstáculos de 2022 e uma alteração com a Carta de Aeródromo de 2026, ambos afirmados pelo DECEA. Fica numa execução à parte (`publicado`), porque é a forma que o trabalho caracteriza.

In [ ]:
RUN_F = al.novo_run(conn, F.AIRAC_2213, "publicado")
e2213, _ = al.escrever(conn, RUN_F, "SBSJ-RWY-PUB", "runway_forma", "BASELINE", corpo=F.FORMA_AIRAC_2213,
                       asserted_by="DECEA", reason="AOC AIRAC AMDT 2213A1")
apoio.avancar_ate(conn, RUN_F, F.AIRAC_2604)
antes, depois = al.ledger.achatar(F.FORMA_AIRAC_2213), al.ledger.achatar(F.FORMA_AIRAC_2604)
e2604, _ = al.escrever(conn, RUN_F, "SBSJ-RWY-PUB", "runway_forma", "DELTA",
                       alteracoes={p: v for p, v in depois.items() if antes[p] != v},
                       asserted_by="DECEA", reason="ADC AIRAC AMDT 2604A1")

E1_DIFF = al.diff(conn, RUN_F, e2213, e2604)
E1_ENVELOPE = al.envelope(conn, e2604)
tabela(["campo", "2022 (AOC)", "2026 (ADC)"], [(f"`{p}`", a, b) for _, p, a, b in E1_DIFF])
print(f"{len(E1_DIFF)} campos mudaram · causa registrada: {E1_ENVELOPE['caused_by']} · motivo: {E1_ENVELOPE['reason']!r}")

Uma única mudança física — a pista passou de 2676 para 2675 m — aparece como **sete números editados separadamente**, porque TORA, TODA, ASDA e LDA estão guardados em vez de calculados. E o registro diz quem publicou, não por que o número mudou.

## E2 — Derivação: guardando só as premissas, os valores publicados se recalculam? Trocar uma premissa obriga a reescrever outro dado?

Agora o SBSJ no modelo proposto (execução `sbsj`). Entram só as **premissas**: as coordenadas das duas cabeceiras do AD 2.12, a largura da pista, a zona de parada e, para cada direção, o deslocamento da cabeceira e se há SWY e CWY. Ausente não é zero: sem essas declarações, a biblioteca recusa TODA, ASDA e LDA em vez de calculá-las. A pista vai de extremidade a extremidade; como as duas direções declaram deslocamento nulo, as extremidades coincidem com as cabeceiras — a pista vai "da THR16 até a THR34" —, e o número 2675 não é escrito em lugar nenhum. A carga do eAIP é uma publicação só, então os cinco eventos entram como um **grupo**: todos ou nenhum (o E6 mostra por que isso importa).

| Papel | Dado | Fonte |
|---|---|---|
| premissa | coordenadas e elevações das cabeceiras | AD 2.12 |
| premissa | largura da pista; dimensões da zona de parada | AD 2.12 |
| premissa | deslocamento das cabeceiras; presença de SWY e CWY | AD 2.13 (lidos da tabela publicada: a comparação de TODA e LDA confere a regra, não essas premissas) |
| norma | variação magnética; regra das distâncias declaradas | ADC; Anexo 14 |
| derivado | comprimento, azimutes, rumos, TORA/TODA/ASDA/LDA | calculado, nunca escrito |

In [ ]:
RUN = al.novo_run(conn, F.EAIP_20_2026, "sbsj")
REF1 = al.publicar_referencial(conn, apoio.referencial())
CORPO1 = al.carregar_referencial(conn, REF1)

AD212 = "eAIP 20-2026, AD 2.12"
CARGA = al.escrever_grupo(conn, RUN, [
    dict(feature_id="SBSJ", feature_type="airport", event_type="BASELINE", asserted_by="DECEA",
         reason="eAIP 20-2026 (AD 2); ADC AIRAC 2604A1",
         corpo={"adc": {"icao": "SBSJ"}, "referencial": {"ref_id": REF1},
                "norma": {"variacao_magnetica_graus": F.VARIACAO_LESTE}}),
    dict(feature_id="SBSJ-RWY", feature_type="runway", event_type="BASELINE", asserted_by="DECEA", reason=AD212,
         corpo={"adc": {"designacao": "16/34", "direcoes": {"inicio": "16", "fim": "34"}, "superficie": F.SUPERFICIE},
                "geometria": {"primitivos": {"pista": {"tipo": "retangulo", "inicio": s("THR16"), "fim": s("THR34"),
                                                       "largura": F.LARGURA_RWY}}}}),
    *[dict(feature_id=f"SBSJ-THR{d}", feature_type="runway_direction", event_type="BASELINE", asserted_by="DECEA",
           reason=F.MOTIVO_DIRECOES[d], corpo={"adc": {"designador": d, "pista": "SBSJ-RWY", **F.DIRECOES[d]},
                                "geometria": {"primitivos": {"thr": {"tipo": "ponto", "seed": s(ref)}}}})
      for d, ref in (("16", "THR16"), ("34", "THR34"))],
    dict(feature_id="SBSJ-SWY", feature_type="protect_area", event_type="BASELINE", asserted_by="DECEA", reason=AD212,
         corpo={"adc": {"papel": "swy", "aplica_a": {"pista": "SBSJ-RWY", "direcao": "16"}},
                "geometria": {"primitivos": {"swy": {"tipo": "retangulo", "inicio": s("THR34"),
                                                     "fim": s("THR34", -F.SWY["comprimento"]),
                                                     "largura": F.SWY["largura"]}}}}),
])
T_A = al.agora(conn, RUN)
print(f"premissas escritas em {T_A}, como um grupo de {len(CARGA)} eventos")

O desenho abaixo **não está no banco**: é gerado na leitura, a partir das cabeceiras e das posições relativas.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for el in adc.projetar(conn, RUN)["elementos"]:
    if el["forma"] is not None:
        x, y = el["forma"].exterior.xy
        ax.fill(x, y, alpha=0.6, label=el["feature_id"])
    for p in el["nos"]:
        ax.plot(p.x, p.y, "k^")
ax.set_aspect("equal"); ax.set_xlabel("leste (m)"); ax.set_ylabel("norte (m)")
ax.set_title("Pista e zona de parada geradas das premissas (plano local na THR16)")
ax.legend(loc="lower left"); plt.show()

### Os valores calculados e os publicados

In [ ]:
L1 = derivacoes.comprimento(CORPO1, "RWY_16_34")
AZ1 = derivacoes.azimutes(CORPO1, "RWY_16_34")
RUMO = {d: derivacoes.rumo_magnetico(AZ1[f"THR{d}"], F.VARIACAO_LESTE) for d in ("16", "34")}
DD1 = al.distancias_declaradas(conn, RUN)

PUBLICADO = {"comprimento": 2675, "azimute 16": 134.93, "azimute 34": 314.92, "rumo 16": 157, "rumo 34": 337,
             "TORA 16": 2675, "TODA 16": 2675, "ASDA 16": 2958, "LDA 16": 2675,
             "TORA 34": 2675, "TODA 34": 2675, "ASDA 34": 2675, "LDA 34": 2675}
CALCULADO = {"comprimento": L1, "azimute 16": AZ1["THR16"], "azimute 34": AZ1["THR34"],
             "rumo 16": RUMO["16"], "rumo 34": RUMO["34"],
             **{f"{k} {d}": DD1[d][k] for d in ("16", "34") for k in ("TORA", "TODA", "ASDA", "LDA")}}
tabela(["grandeza", "calculado", "publicado", "diferença"],
       [(g, f"{CALCULADO[g]:.3f}", PUBLICADO[g], f"{CALCULADO[g] - PUBLICADO[g]:+.3f}") for g in PUBLICADO])

E2_DERIVADOS_NO_BANCO = apoio.contar(conn, r'''SELECT count(*) FROM event_field f JOIN event e USING (event_id)
    WHERE e.run_id = %s AND regexp_replace(f.path, '^.*\.', '') IN
          ('tora','toda','asda','lda','azimute','rumo','comprimento')''', RUN)
print("campos de valor derivado guardados no banco:", E2_DERIVADOS_NO_BANCO)

### A cabeceira 34 é remedida

Cenário simulado: um novo levantamento desloca a THR34 em cerca de 1 m. O levantamento entra como evento próprio, e a troca de referencial aponta para ele como causa. Nenhuma posição de pista ou de zona de parada é reescrita.

In [ ]:
al.avancar(conn, RUN, dias=30)
MARCO = apoio.ultimo_event_id(conn, RUN)
LEV, _ = al.escrever(conn, RUN, "SBSJ-LEV-THR34", "survey", "BASELINE",
                     asserted_by="Heyder (cenário simulado)", reason="remedição da cabeceira 34",
                     corpo={"levantamento": {"ponto": "THR34", "lat_publicada": F.THR34_REMEDIDA["lat"],
                                             "lon_publicada": F.THR34_REMEDIDA["lon"]}})
REF2 = al.publicar_referencial(conn, apoio.referencial(F.THR34_REMEDIDA, "levantamento SBSJ-LEV-THR34"))
CORPO2 = al.carregar_referencial(conn, REF2)
TROCA, _ = al.escrever(conn, RUN, "SBSJ", "airport", "DELTA", alteracoes={"referencial.ref_id": REF2},
                       asserted_by="DECEA", reason="troca de referencial: THR34 remedida", caused_by=LEV)
E2_ESCRITOS = apoio.escritos_depois_de(conn, RUN, MARCO)
tabela(["feature", "tipo", "evento", "campos escritos"],
       [(f, t, et, ", ".join(p)) for _, f, t, et, p in E2_ESCRITOS])

L2 = derivacoes.comprimento(CORPO2, "RWY_16_34")
AZ2 = derivacoes.azimutes(CORPO2, "RWY_16_34")
print(f"comprimento: {L1:.3f} m → {L2:.3f} m, sem nenhum número digitado")

**Conferência com o gabarito.** Os cantos da pista e da zona de parada, calculados pela biblioteca com o referencial antigo e com o novo, comparados com o mesmo cálculo feito por outro código (fórmulas de Vincenty, `geodesia.py`).

In [ ]:
E2_VERTICES = []
for nome, corpo in (("antes", CORPO1), ("depois", CORPO2)):
    res = Resolvedor(corpo)
    for ref, s_, o_ in F.VERTICES:
        outro = "THR34" if ref == "THR16" else "THR16"
        lat, lon, _ = res.seed(s(ref, s_, o_))
        glat, glon = gabarito.resolver(apoio.latlon(corpo, ref), apoio.latlon(corpo, outro), s_, o_)
        E2_VERTICES.append((nome, f"{ref} s={s_} o={o_}", gabarito.inversa(lat, lon, glat, glon)[0]))
tabela(["referencial", "canto", "distância biblioteca × gabarito (m)"], [(a, b, f"{c:.2e}") for a, b, c in E2_VERTICES])

gL, gaz16, gaz34 = gabarito.inversa(*apoio.latlon(CORPO2, "THR16"), *apoio.latlon(CORPO2, "THR34"))
E2_DL = abs(L2 - gL)
E2_DAZ = max(abs(AZ2["THR16"] - gaz16), abs(AZ2["THR34"] - gaz34))
print(f"comprimento novo: biblioteca {L2:.4f} m · gabarito {gL:.4f} m · Δ = {E2_DL:.1e} m; maior Δ de azimute {E2_DAZ:.1e}°")

## E3 — Reconstrução no tempo: o histórico basta para reconstruir qualquer instante, com autor, motivo e causa?

As perguntas de um auditor, respondidas só lendo o banco:

- **estado em *t***: antes da remedição vale o referencial antigo e o comprimento antigo; agora, os novos;
- **ciclo de vida**: uma interdição temporária de 300 m a partir da THR34 entra num dia e sai três dias depois, sem nenhum tipo especial de "temporário";
- **quem e por quê**: o envelope da troca de referencial dá autor, motivo e causa;
- **o número está certo**: as distâncias recalculadas para o passado são as que valiam no passado.

In [ ]:
al.avancar(conn, RUN, dias=10)
T4 = al.agora(conn, RUN)
B4, _ = al.escrever(conn, RUN, "SBSJ-RES-0001", "restriction", "BASELINE", owner_ref="SBSJ-RWY",
                    asserted_by="operador SBSJ", reason="NOTAM (simulado)",
                    corpo={"adc": {"texto": "RWY 16/34: 300 m a partir da THR 34 interditados para obra",
                                   "trecho": {"de": s("THR34"), "ate": s("THR34", 300)}}})
al.avancar(conn, RUN, dias=3)
T5 = al.agora(conn, RUN)
V5, _ = al.escrever(conn, RUN, "SBSJ-RES-0001", "restriction", "VOID", owner_ref="SBSJ-RWY",
                    asserted_by="operador SBSJ", reason="obra concluída")
al.avancar(conn, RUN, dias=1)
AGORA = al.agora(conn, RUN)

ref_a, corpo_a = al.referencial_vigente(conn, RUN, al.relogio.limite(T_A))
ref_b, corpo_b = al.referencial_vigente(conn, RUN, al.relogio.limite(AGORA))
E3_ESTADO = {"t_a": (ref_a, derivacoes.comprimento(corpo_a, "RWY_16_34")),
             "agora": (ref_b, derivacoes.comprimento(corpo_b, "RWY_16_34"))}
tabela(["instante", "referencial vigente", "comprimento derivado (m)"],
       [(k, f"`{r[:12]}…`", f"{L:.3f}") for k, (r, L) in E3_ESTADO.items()])

def interditada(t):
    return any(f == "SBSJ-RES-0001" for f, _ in al.features(conn, RUN, t, tipos=["restriction"]))

E3_DENTRO = [T4, T4 + timedelta(days=1.5), apoio.um_segundo_antes(T5)]
E3_FORA = [apoio.um_segundo_antes(T4), T5, AGORA]
tabela(["instante", "interditada?"], [(f"{t:%d/%m %H:%M:%S}", interditada(t)) for t in sorted(E3_DENTRO + E3_FORA)])

E3_CICLOS = al.ciclos(conn, RUN, "SBSJ-RES-0001")
E3_ENV_TROCA, E3_ENV_LEV = al.envelope(conn, TROCA), al.envelope(conn, LEV)
print("ciclo de vida da interdição:", [(tipo, ciclo, f"{al.instante(e):%d/%m}") for e, tipo, ciclo in E3_CICLOS])
print(f"troca de referencial: afirmada por {E3_ENV_TROCA['asserted_by']}, motivo '{E3_ENV_TROCA['reason']}', "
      f"causa {E3_ENV_LEV['feature_id']} (afirmada por {E3_ENV_LEV['asserted_by']})")

E3_DD_TA = al.distancias_declaradas(conn, RUN, T_A)
E3_TRILHA = al.diff(conn, RUN, T_A, AGORA)
display(Markdown("**Trilha de auditoria entre t_a e agora**"))
tabela(["feature", "campo", "antes", "depois"], [(f, f"`{p}`", str(a)[:20], str(b)[:20]) for f, p, a, b in E3_TRILHA])

## E4 — Adulteração: uma alteração direto no banco é detectada? Qual escapa, e o que a detecta?

O PANS-AIM (2.2.1) recomenda garantir a integridade dos dados com criptografia, como funções de hash. Aqui o atacante tem acesso direto ao banco e não passa pela biblioteca. Cada ataque roda numa **cópia** do banco. Antes, anota-se fora do banco o hash do último evento: a **âncora**.

In [ ]:
E4_ANCORA = al.cabeca(conn, RUN)
E4_K, E4_K_SEGUINTE, _ = ataques.escolher_k(conn, RUN)
conn.close()                               # o PostgreSQL só copia um banco sem conexões abertas

E4 = {}
for ataque in ataques.ATAQUES:
    c2 = apoio.copia(apoio.BANCO, "aeroledger_ataque")
    ataques.atacar(c2, RUN, ataque, E4_K)
    E4[ataque] = ataques.detectar(c2, RUN, E4_ANCORA)
    c2.close(); apoio.apagar("aeroledger_ataque")

conn = apoio.reconectar()
print(f"âncora {E4_ANCORA[1][:16]}… · evento atacado k = {E4_K}")
tabela(["ataque", "verificar_cadeia aponta", "com a âncora"],
       [(ataques.ATAQUES[a], vc or "nada", "detecta" if ca else "não detecta") for a, (vc, ca) in E4.items()])

A última linha é o limite: quem controla o banco inteiro e recalcula todos os hashes passa pela verificação interna. Só a âncora, guardada fora do banco, o denuncia.

**Limites que este trabalho não testa.** O banco registra quem afirmou, mas não prova (não há assinatura digital por evento). E nenhum sistema torna uma medição verdadeira: se a fonte mede errado, o banco só atribui o erro a quem mediu — como as edições que divergem no E1.

## E5 — Consistência e verificação: produtos do mesmo banco podem divergir? Um terceiro verifica a integridade sem a biblioteca?

O PANS-AIM (2.1.3.2) exige consistência entre os produtos de informação aeronáutica. No E1, os produtos reais divergem. Aqui, dois produtos saem do mesmo banco:

- **P1**, a tabela do AIP: comprimento e distâncias declaradas;
- **P2**, o conteúdo da carta de aeródromo (a lente ADC).

Como os dois saem da mesma derivação, a consistência vem por construção. A parte que não é óbvia vem depois: um **verificador independente** (`verificador_independente.py`), escrito só a partir da especificação e sem importar a biblioteca, confere a cadeia e o comprimento a partir de um JSON exportado.

In [ ]:
def p1(t):
    _, corpo = al.referencial_vigente(conn, RUN, al.relogio.limite(t))
    return {"comprimento": derivacoes.comprimento(corpo, "RWY_16_34"), "distancias": al.distancias_declaradas(conn, RUN, t)}

def p2(t):
    return {"distancias": adc.projetar(conn, RUN, t)["distancias"]}

E5 = {"P1 t_a": p1(T_A), "P2 t_a": p2(T_A), "P1 agora": p1(AGORA), "P2 agora": p2(AGORA)}
tabela(["produto", "ASDA 16 (m)", "LDA 34 (m)"],
       [(k, f"{v['distancias']['16']['ASDA']:.3f}", f"{v['distancias']['34']['LDA']:.3f}") for k, v in E5.items()])

EXPORT = Path(tempfile.mkdtemp()) / "sbsj.json"
apoio.exportar(conn, RUN, EXPORT)
dados = json.loads(EXPORT.read_text())
E5_VERIFICADOR = vi.verificar(dados)
E5_CABECA = list(al.cabeca(conn, RUN))
E5_L_VERIFICADOR = vi.comprimento(dados)
print(f"verificador independente: íntegra = {E5_VERIFICADOR['integra']}, "
      f"cabeça {E5_VERIFICADOR['cabeca'][1][:16]}… (biblioteca: {E5_CABECA[1][:16]}…)")
print(f"comprimento: verificador {E5_L_VERIFICADOR:.4f} m · biblioteca {E5['P1 agora']['comprimento']:.4f} m")

E5_ALVO = next(e for e in dados["eventos"] if e["feature_id"] == "SBSJ-RWY")
for campo in E5_ALVO["campos"]:
    if campo["path"].endswith("pista.largura"):
        campo["next"] += 1
E5_ADULTERADO = vi.verificar(dados)
print(f"largura da pista alterada no JSON → o verificador acusa o evento {E5_ADULTERADO['primeiro_divergente']}")

## E6 — Leitura separada: quem lê sem acesso ao banco chega à mesma resposta, nunca vê uma mudança pela metade e percebe um canal adulterado?

Até aqui toda leitura foi feita no próprio banco. Na arquitetura proposta, escrita e leitura se separam (CQRS): o ledger continua a única fonte de verdade e recebe a escrita, que precisa ser consistente; a leitura sai dele. O ledger é publicado num **tópico geral** — o log ordenado do run, como CDC —, e cada **consumidor** lê tudo, no próprio ritmo, confere a cadeia mensagem a mensagem e monta do lado dele só o que precisa. O PANS-AIM deixa a distribuição em aberto (2.1.4, "*to be developed*"); a pergunta é se ela pode ser feita sem perder o que E3, E4 e E5 mostraram.

Uma pergunta por propriedade:

- **equivalência** — o consumidor responde, em cada corte do histórico, o mesmo que o banco?
- **atomicidade** — algum corte mostra a carga do eAIP pela metade?
- **recorte** — quem assina só a cabeceira 16 chega aos mesmos derivados, lendo menos?
- **limite explícito e disponibilidade** — quem está atrasado recusa em vez de responder velho? Com o banco fora, a leitura continua?
- **determinismo** — consumidores com cadências diferentes, um que caiu e retomou e um que leu um tópico refeito do zero chegam ao mesmo hash no mesmo limite?
- **canal adulterado** — uma mensagem alterada, removida ou inserida no tópico é percebida? Qual escapa, e o que a detecta?

O tópico aqui é em memória; um broker como o Kafka cumpriria o mesmo contrato (anexar, e ler a partir de um offset). Toda leitura de consumidor leva um limite explícito (`event_id` ou instante): não existe "agora".

In [ ]:
T6 = al.topico
TOPICO = T6.TopicoMemoria()
E6_PUBLICADOS = T6.retransmitir(conn, RUN, TOPICO)          # CDC: ledger → tópico
TUDO = T6.Consumidor(TOPICO, nome="lê tudo")
TUDO.consumir()
print(f"{E6_PUBLICADOS} eventos no tópico ({TOPICO.fim()} mensagens, contando os corpos dos referenciais); "
      f"consumidor em {TUDO.cabeca[0]}")

# Fronteiras: onde o histórico pode ser cortado (eventos fora de grupo e o último de cada grupo).
EVENTOS = T6.eventos_do_ledger(conn, RUN)
ULTIMO_DO_GRUPO = {}
for e in EVENTOS:
    if e["grupo_id"] is not None:
        ULTIMO_DO_GRUPO[e["grupo_id"]] = e["event_id"]
FRONTEIRAS = [e["event_id"] for e in EVENTOS if e["grupo_id"] is None or ULTIMO_DO_GRUPO[e["grupo_id"]] == e["event_id"]]

def no_banco(t):
    """O que o banco responde em t: estados das features vivas e distâncias declaradas."""
    estados = {f: {"tipo": tp, "estado": al.estado(conn, RUN, f, t)} for f, tp in al.features(conn, RUN, t)}
    return estados, al.distancias_declaradas(conn, RUN, t)

E6_EQUIV = []
CORTES = [(f"evento {n}", n) for n in FRONTEIRAS] + [
    ("t_a (instante)", T_A), ("interdição em vigor (instante)", T4 + timedelta(days=1.5)),
    ("1 s antes do fim da interdição", apoio.um_segundo_antes(T5))]
for rotulo, t in CORTES:
    r, (est, dist) = TUDO.ler(t), no_banco(t)
    E6_EQUIV.append((rotulo, r["estados"] == est, r["derivados"]["distancias"] == dist))
tabela(["corte", "estados iguais ao banco", "distâncias iguais ao banco"], E6_EQUIV)

### Atomicidade: um corte no meio da carga

O consumidor só corta em fronteira de grupo, e a leitura direta do banco, desde a `aeroledger` 0.7.0, também: um limite que cai dentro do grupo recua para antes dele. Só a primitiva SQL (`estado_em`), que não conhece grupos, ainda vê a carga pela metade. O que ela mostraria — a pista sem as cabeceiras e sem a zona de parada — não vira um número errado: sem as cabeceiras, a regra recusa TODA, ASDA e LDA.

In [ ]:
MEIO = CARGA[1][0]                     # logo depois da pista, antes das cabeceiras e da zona de parada
E6_LIM_BANCO = al.ledger._limite(conn, RUN, MEIO)
try:
    E6_BANCO_MEIO = al.distancias_declaradas(conn, RUN, MEIO)
except ValueError as exc:
    E6_BANCO_MEIO = {"recusado": str(exc)}
E6_CONS_MEIO = TUDO.ler(MEIO)
E6_MEIOS = {e: TUDO.ler(e)["ate"] for e, _ in CARGA[:-1]}
print(f"banco cortado no evento {MEIO}: limite efetivo {E6_LIM_BANCO} (antes da carga, que começa em {CARGA[0][0]}); "
      f"distâncias: {E6_BANCO_MEIO}")
print(f"consumidor, mesmo pedido: corte efetivo {E6_CONS_MEIO['ate']} (antes da carga), "
      f"{len(E6_CONS_MEIO['estados'])} features, derivados {E6_CONS_MEIO['derivados']}")

# A primitiva SQL, sem a noção de grupo, vê a pista no meio da carga...
E6_PRIMITIVA = apoio.contar(conn, "SELECT count(*) FROM estado_em(%s, 'SBSJ-RWY', %s)", RUN, MEIO)
# ...e, se alguém montasse esse estado parcial, a regra recusaria em vez de dar ASDA 16 = TORA 16.
PISTA = {c: v for c, v in al.estado(conn, RUN, "SBSJ-RWY", CARGA[-1][0]).items() if c in ("adc", "geometria")}
E6_PARCIAL = derivacoes.distancias_de(CORPO1, {"SBSJ-RWY": ("runway", PISTA)})["16"]
print(f"primitiva SQL no mesmo corte: {E6_PRIMITIVA} campos da pista visíveis; "
      f"regra sobre esse estado parcial: {E6_PARCIAL}")


### Recorte: um consumidor que assina só a cabeceira 16

Ele recebe o mesmo tópico, mas dobra só o fecho de dependência da THR16: as features ligadas a ela por referência declarada (a pista, por `adc.pista`; a outra cabeceira e a zona de parada, que apontam para a pista) e o `airport`, porque toda posição se apoia no referencial.

In [ ]:
N = TUDO.cabeca[0]
SUB = T6.Consumidor(TOPICO, raizes=["SBSJ-THR16"], nome="só THR16")
SUB.consumir()
R_TUDO, R_SUB = TUDO.ler(N), SUB.ler(N)
tabela(["consumidor", "features dobradas", "hash dos derivados"],
       [(c, len(r["fecho"]), f"`{r['hash_derivados'][:16]}…`") for c, r in (("lê tudo", R_TUDO), ("só THR16", R_SUB))])
print("fora do recorte:", sorted(set(R_TUDO["fecho"]) - set(R_SUB["fecho"])))

### Ao vivo: limite explícito e banco fora

Um NOTAM novo é escrito no banco. Antes de recebê-lo, o consumidor é perguntado sobre ele; depois, o banco é fechado e o consumidor é perguntado de novo.

In [ ]:
al.avancar(conn, RUN, dias=1)
NOVO, _ = al.escrever(conn, RUN, "SBSJ-RES-0002", "restriction", "BASELINE", owner_ref="SBSJ-RWY",
                      asserted_by="operador SBSJ", reason="NOTAM (simulado)",
                      corpo={"adc": {"texto": "RWY 16/34: sinalização horizontal em manutenção"}})
try:
    TUDO.ler(NOVO)
    E6_ATRASADO = None
except T6.ConsumidorAtrasado as exc:
    E6_ATRASADO = str(exc)
print("antes de receber:", E6_ATRASADO)

T6.retransmitir(conn, RUN, TOPICO)
TUDO.consumir()
E6_DEPOIS = TUDO.ler(NOVO)
print("depois de receber:", "SBSJ-RES-0002" in E6_DEPOIS["estados"], "· corte", E6_DEPOIS["ate"])

N_FINAL = NOVO
E6_ANTES = TUDO.ler(N_FINAL)
conn.close()                                 # banco fora
E6_SEM_BANCO = TUDO.ler(N_FINAL)
print("com o banco fechado, mesma resposta:", E6_SEM_BANCO == E6_ANTES)
conn = apoio.reconectar()

### Determinismo: seis consumidores, mesmo limite

Cada um lê o tópico de um jeito: em laço, uma mensagem por vez; em lotes de duas (menores que a carga, que tem cinco eventos); por cron, de uma vez; um que cai no meio, grava o checkpoint e retoma; e um que lê um tópico refeito do zero a partir do banco. Todos são perguntados no mesmo limite.

In [ ]:
def ate_o_fim(c, passo=None):
    while not c.em_dia():
        c.consumir(passo)
    return c

LACO = ate_o_fim(T6.Consumidor(TOPICO, nome="laço, uma por vez"), 1)
LOTES = ate_o_fim(T6.Consumidor(TOPICO, nome="lotes de duas"), 2)
CRON = ate_o_fim(T6.Consumidor(TOPICO, nome="cron, de uma vez"))
QUEDA = T6.Consumidor(TOPICO, nome="caiu e retomou")
QUEDA.consumir(4)
E6_CHECKPOINT = QUEDA.salvar(Path(tempfile.mkdtemp()) / "checkpoint.json")
del QUEDA                                    # o processo cai; só o arquivo sobrevive
QUEDA = ate_o_fim(T6.Consumidor.carregar(E6_CHECKPOINT, TOPICO))
REFEITO = T6.TopicoMemoria()
T6.retransmitir(conn, RUN, REFEITO)
ZERO = ate_o_fim(T6.Consumidor(REFEITO, nome="tópico refeito do zero"))

E6_HASHES = {c.nome: (c.ler(N_FINAL)["hash"], c.ler(T_A)["hash"]) for c in (TUDO, LACO, LOTES, CRON, QUEDA, ZERO)}
tabela(["consumidor", f"hash em {N_FINAL}", "hash em t_a"],
       [(k, f"`{a[:16]}…`", f"`{b[:16]}…`") for k, (a, b) in E6_HASHES.items()])

### Canal adulterado

Agora o atacante está no canal (o broker), não no banco: o ledger não é tocado. Cada ataque roda num tópico próprio, refeito do banco. O alvo *k* é a troca de referencial da remedição; alterar é fazer o tópico dizer que o referencial adotado continua o antigo. Para cada ataque: o consumidor para em alguma mensagem? Ficou fiel ao banco (nenhum evento aplicado difere do ledger)? A cabeça dele bate com a âncora (a cabeça do ledger, guardada fora do canal)?

In [ ]:
E6_ANCORA = tuple(al.cabeca(conn, RUN))
HASH_NO_LEDGER = {e["event_id"]: e["hash"] for e in T6.eventos_do_ledger(conn, RUN)}
E6_ATAQUES = {}
for ataque in ataques.ATAQUES_TOPICO:
    top = T6.TopicoMemoria()
    T6.retransmitir(conn, RUN, top)
    ataques.atacar_topico(top, ataque, TROCA, "referencial.ref_id", REF1)
    c = T6.Consumidor(top, nome=ataque)
    try:
        c.consumir()
        erro = None
    except T6.TopicoCorrompido as exc:
        erro = exc
    E6_ATAQUES[ataque] = {"para": erro is not None, "motivo": erro.motivo if erro else "—",
                          "fiel": all(HASH_NO_LEDGER.get(e["event_id"]) == e["hash"] for e in c.mensagens),
                          "na_ancora": tuple(c.cabeca) == E6_ANCORA}
tabela(["ataque", "consumidor para?", "fiel ao banco?", "cabeça = âncora?", "motivo"],
       [(ataques.ATAQUES_TOPICO[a], r["para"], r["fiel"], r["na_ancora"], r["motivo"]) for a, r in E6_ATAQUES.items()])

No meio do tópico, toda adulteração para o consumidor, e ele fica num prefixo fiel ao banco: quando o elo não fecha, não há como saber se o falso é a mensagem recebida ou a última aplicada, então ele volta ao ponto comum ou descarta a última. As duas últimas linhas são o limite, o mesmo do E4: o encadeamento prova que nada do que chegou foi alterado, mas não que chegou tudo. Um tópico cortado no fim parece só atrasado, e um evento falso anexado no fim passa pelo elo. Os dois só aparecem comparando a cabeça do consumidor com a âncora, publicada fora do canal (por exemplo, junto de cada ciclo AIRAC).

**O que fica fora.** O transporte real (Kafka) e o custo em escala não são medidos; tópicos por nó, confidencialidade (quem assina recebe tudo) e mais de um aeroporto (uma cadeia por run) ficam argumentados, não testados.

---

# Verificações

Cada célula abaixo responde à pergunta de um experimento com `assert`. O valor esperado e a margem estão na própria linha, com a origem no comentário. As margens nunca foram escolhidas: saem da precisão com que o AIP publica cada número ou da exatidão do gabarito.

Se uma verificação falhar, os resultados acima continuam visíveis, e a mensagem diz o valor observado.

In [ ]:
# E1 — causa (caracterização): o registro das edições publicadas NÃO registra por que o valor mudou
# A pista, a SWY e a faixa mudaram entre as edições; em "modo forma" isso aparece como
# 12 campos: comprimento da pista, SWY, comprimento e largura da faixa, e as 8 distâncias declaradas.
assert len(E1_DIFF) == 12, f"{len(E1_DIFF)} campos"
assert {p.split(".")[0] for _, p, _, _ in E1_DIFF} == {"pista", "swy", "faixa", "distancias"}
# Nenhuma mudança traz a causa: só a edição que a publicou.
assert E1_ENVELOPE["caused_by"] is None
print("E1 ok")

In [ ]:
# E2 — derivação: sim, os valores se recalculam das premissas, e a troca de premissa não reescreve nada
# Margens pela precisão de publicação (PANS-AIM, App. 2):
#   comprimento e distâncias: 0,5 m (publicados ao metro) + 0,419 m (coordenadas das cabeceiras a 0,01")
#   ASDA 16: o anterior + 0,5 m da SWY, publicada ao metro
#   azimute: 0,005° (publicado a 0,01°) + 0,009° (efeito das coordenadas no rumo)
#   rumo magnético: 0,5° (rumo inteiro) + 0,5° (variação inteira) + 0,009°
MARGEM = {"comprimento": 0.919, "azimute": 0.014, "rumo": 1.009, "distancia": 0.919, "ASDA 16": 1.419}

def margem(g):
    if g == "ASDA 16":
        return MARGEM["ASDA 16"]
    return MARGEM[g.split()[0]] if g.split()[0] in MARGEM else MARGEM["distancia"]

for g, pub in PUBLICADO.items():
    dif = abs(CALCULADO[g] - pub)
    if g.startswith(("azimute", "rumo")):
        dif = min(dif, 360 - dif)
    assert dif <= margem(g), f"{g}: calculado {CALCULADO[g]:.3f}, publicado {pub}, |Δ| {dif:.3f} > {margem(g)}"

# Nenhuma distância foi recusada: as direções declaram deslocamento, SWY e CWY.
assert not any("recusado" in v for v in DD1.values()), DD1

# Nenhum valor derivado foi guardado no banco.
assert E2_DERIVADOS_NO_BANCO == 0

# A remedição escreveu só o levantamento e a troca de referencial, com o levantamento como causa.
assert [(f, et) for _, f, _, et, _ in E2_ESCRITOS] == [("SBSJ-LEV-THR34", "BASELINE"), ("SBSJ", "DELTA")]
assert E2_ESCRITOS[1][4] == ["referencial.ref_id"]
assert al.envelope(conn, TROCA)["caused_by"] == LEV
# Nenhuma feature com posição (pista, cabeceiras, SWY) foi reescrita.
assert not any(t in ("runway", "runway_direction", "protect_area") for _, _, t, _, _ in E2_ESCRITOS)

# A biblioteca coincide com o gabarito (outro código) dentro da exatidão de Vincenty: 0,5 mm.
assert max(d for _, _, d in E2_VERTICES) <= 0.0005
assert E2_DL <= 0.0005
assert E2_DAZ <= 0.0005 / L2 * 180 / 3.141592653589793      # 0,5 mm na ponta da pista, em graus
print("E2 ok")

In [ ]:
# E3 — reconstrução no tempo: sim, qualquer instante, com autor, motivo e causa
# Antes da remedição vale o referencial antigo e o comprimento antigo; agora, os novos.
assert E3_ESTADO["t_a"] == (REF1, L1) and E3_ESTADO["agora"] == (REF2, L2)
# A interdição existe em [t4, t5) e não existe fora.
assert all(interditada(t) for t in E3_DENTRO) and not any(interditada(t) for t in E3_FORA)
# O ciclo de vida é exatamente: criada em t4, encerrada em t5.
assert E3_CICLOS == [(B4, "BASELINE", 1), (V5, "VOID", 1)]
assert al.instante(B4) == T4 and al.instante(V5) == T5
# O envelope responde quem, por quê e com qual causa.
assert E3_ENV_TROCA["asserted_by"] == "DECEA" and E3_ENV_TROCA["reason"] and E3_ENV_TROCA["caused_by"] == LEV
assert E3_ENV_LEV["asserted_by"]
# As distâncias recalculadas para t_a são, bit a bit, as que valiam em t_a.
assert E3_DD_TA == DD1
print("E3 ok")

In [ ]:
# E4 — adulteração: detectada no meio da cadeia; a reescrita total escapa ao banco e só a âncora a detecta
# Alterar, apagar ou inserir no meio: a verificação interna aponta o evento.
assert E4["alterar"] == ([E4_K], True)              # aponta o próprio k
assert E4["apagar"] == ([E4_K_SEGUINTE], True)      # o seguinte perde o elo com o anterior
assert E4["inserir"] == ([E4_K_SEGUINTE], True)     # idem: o seguinte não aponta para o inserido
# Reescrever tudo de k até o fim: a verificação interna não percebe; só a âncora externa.
assert E4["reescrever"] == ([], True)
print("E4 ok")

In [ ]:
# E5 — consistência e verificação: os produtos não divergem, e um terceiro verifica sem a biblioteca
# Os dois produtos dão as mesmas distâncias, antes e depois da remedição...
assert E5["P1 t_a"]["distancias"] == E5["P2 t_a"]["distancias"]
assert E5["P1 agora"]["distancias"] == E5["P2 agora"]["distancias"]
# ...e mudam juntos.
assert E5["P1 t_a"]["distancias"] != E5["P1 agora"]["distancias"]
# O verificador independente confirma a cadeia e chega ao mesmo último hash.
assert E5_VERIFICADOR["integra"] and list(E5_VERIFICADOR["cabeca"]) == E5_CABECA
# E o mesmo comprimento, dentro da exatidão de Vincenty (0,5 mm).
assert abs(E5_L_VERIFICADOR - E5["P1 agora"]["comprimento"]) <= 0.0005
# Alterado um campo do arquivo exportado, ele aponta o evento alterado.
assert E5_ADULTERADO["primeiro_divergente"] == E5_ALVO["event_id"]
print("E5 ok")

In [ ]:
# E6 — leitura separada: mesma resposta, nunca pela metade, canal adulterado percebido (com o limite do fim)
# Equivalência: em cada fronteira e em cada instante testado, o consumidor dá os mesmos estados e distâncias que o banco.
assert all(est and dist for _, est, dist in E6_EQUIV), [c for c in E6_EQUIV if not (c[1] and c[2])]
# Atomicidade: nem o banco nem o consumidor cortam no meio da carga; os dois recuam para antes dela
# e respondem o mesmo (sem referencial vigente, sem derivados).
assert E6_LIM_BANCO < CARGA[0][0]
assert all(a is None for a in E6_MEIOS.values()) and E6_CONS_MEIO["estados"] == {}
assert E6_BANCO_MEIO == E6_CONS_MEIO["derivados"] == {"recusado": "sem referencial vigente"}
# Só a primitiva SQL vê a carga pela metade, e a regra, sobre esse estado parcial, recusa em vez de dar ASDA = TORA.
assert E6_PRIMITIVA > 0
assert "ASDA" not in E6_PARCIAL and "ASDA" in E6_PARCIAL["recusado"] and abs(E6_PARCIAL["TORA"] - DD1["16"]["TORA"]) < 1e-9
# Recorte: quem assina só a THR16 dobra menos features e chega aos mesmos derivados e aos mesmos estados.
assert set(R_SUB["fecho"]) < set(R_TUDO["fecho"]) and R_SUB["hash_derivados"] == R_TUDO["hash_derivados"]
assert all(R_SUB["estados"][f] == R_TUDO["estados"][f] for f in R_SUB["fecho"])
# Limite explícito: atrasado, recusa; depois de receber, responde com o evento novo.
assert E6_ATRASADO is not None and "SBSJ-RES-0002" in E6_DEPOIS["estados"] and E6_DEPOIS["ate"] == NOVO
# Disponibilidade: com o banco fechado, a mesma resposta.
assert E6_SEM_BANCO == E6_ANTES
# Determinismo: seis consumidores, o mesmo hash no mesmo limite (no fim e em t_a).
assert len(E6_HASHES) == 6 and len(set(E6_HASHES.values())) == 1
# Canal adulterado no meio do tópico: o consumidor para e fica fiel ao banco.
for a in ("alterar", "alterar_rehash", "remover", "inserir"):
    assert E6_ATAQUES[a]["para"] and E6_ATAQUES[a]["fiel"] and not E6_ATAQUES[a]["na_ancora"], (a, E6_ATAQUES[a])
# No fim do tópico, o elo não percebe: cortar parece atraso; anexar um falso é aplicado. Só a âncora detecta.
assert E6_ATAQUES["truncar_fim"] == {"para": False, "motivo": "—", "fiel": True, "na_ancora": False}
assert E6_ATAQUES["anexar_fim"] == {"para": False, "motivo": "—", "fiel": False, "na_ancora": False}
print("E6 ok")